# Live Demo Pipeline — Auditable Wikipedia RAG-ETL

## Live Demo Setup

This demo uses a small subset (~40 docs) but **identical code and schema** as the full pipeline.

### What this notebook demonstrates:

1. **Full rebuild** on initial 40-doc sample → establishes baseline
2. **Idempotency test** → run same full build again, audit confirms zero drift
3. **Augmented build** → add 1 new doc, revise 1 doc, submit 1 duplicate and 1 corrupted doc
4. **Retrieval before/after** → show that revised doc surfaces in results for the new query

### Data preparation
Run `make_live_demo_data.py` to create:
- `data/live_demo/initial_sample.jsonl` — 40 clean Wikipedia docs
- `data/live_demo/update_sample.jsonl` — 4 docs: 1 new, 1 revised, 1 duplicate, 1 corrupted

## Setup

In [ ]:
import os
import subprocess
import duckdb
import pandas as pd

# Navigate to project root
os.chdir('..')
print(f"Working directory: {os.getcwd()}")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

DEMO_DB = 'outputs/live_demo/wiki_demo.duckdb'
DEMO_OUTPUT = 'outputs/live_demo'
DEMO_INDEX = 'outputs/live_demo/faiss/index.faiss'

print("Setup complete.")

## Initial Data Size

In [ ]:
initial_path = 'data/live_demo/initial_sample.jsonl'
update_path = 'data/live_demo/update_sample.jsonl'

if os.path.exists(initial_path):
    with open(initial_path) as f:
        n_initial = sum(1 for line in f if line.strip())
    print(f"initial_sample.jsonl: {n_initial} documents")
else:
    print(f"File not found: {initial_path}")
    print("Run: python src/make_live_demo_data.py --full-input data/full/raw_wiki.jsonl")

if os.path.exists(update_path):
    with open(update_path) as f:
        n_update = sum(1 for line in f if line.strip())
    print(f"update_sample.jsonl: {n_update} documents")
else:
    print(f"File not found: {update_path}")

## STEP 1: Full Rebuild (First Time)

Run the full ETL pipeline on the 40-doc initial sample.

Expected: All 40 docs → NEW, no UPDATED or UNCHANGED.

In [ ]:
print("Running STEP 1: Full rebuild on initial_sample.jsonl...")
result1 = subprocess.run(
    [
        'python', 'src/build.py',
        '--mode', 'full',
        '--dataset-name', 'live_demo',
        '--input', 'data/live_demo/initial_sample.jsonl',
        '--output', DEMO_OUTPUT,
        '--db', DEMO_DB,
        '--index-type', 'ivf'
    ],
    check=True,
    capture_output=False
)
print(f"\nReturn code: {result1.returncode}")

## Runs Table (after Step 1)

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
df_runs = conn.execute("SELECT * FROM runs ORDER BY run_id").df()
print(f"Runs so far: {len(df_runs)}")
display(df_runs[['run_id', 'mode', 'status', 'raw_doc_count', 'new_doc_count',
                  'updated_doc_count', 'unchanged_doc_count', 'active_doc_count',
                  'new_chunk_count', 'active_chunk_count', 'index_vector_count']])
conn.close()

## Row Count Reconciliation (after Step 1)

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
df_recon = conn.execute("SELECT * FROM row_count_reconciliation ORDER BY run_id").df()
print("Row count reconciliation:")
display(df_recon)
conn.close()

## STEP 2: Retrieval Before Update

Run two queries before the augmented update to establish a baseline.

In [ ]:
from src.retrieve import retrieve

# Load the revised doc title from the initial sample
import json
with open('data/live_demo/initial_sample.jsonl') as f:
    docs = [json.loads(line) for line in f if line.strip()]

revised_title = docs[0].get('title', 'the document')

pre_queries = [
    f"What is {revised_title}?",
    "What document discusses vector databases and retrieval systems?"
]

print("=== RETRIEVAL BEFORE AUGMENTED UPDATE ===")
for q in pre_queries:
    print(f"\nQuery: {q}")
    df_r = retrieve(q, DEMO_DB, DEMO_INDEX, top_k=3)
    if df_r.empty:
        print("  No results.")
    else:
        for _, row in df_r.iterrows():
            print(f"  [{row['rank']}] {row['title']} (score={row['score']:.4f})")

## STEP 3: Full Rebuild (AGAIN — Idempotency Test)

Run the exact same full build command again.

Expected: All docs → UNCHANGED. No new chunks or embeddings created.
The audit should confirm PASS on all 5 checks.

In [ ]:
print("Running STEP 3: Full rebuild (idempotency test) on initial_sample.jsonl...")
result3 = subprocess.run(
    [
        'python', 'src/build.py',
        '--mode', 'full',
        '--dataset-name', 'live_demo',
        '--input', 'data/live_demo/initial_sample.jsonl',
        '--output', DEMO_OUTPUT,
        '--db', DEMO_DB,
        '--index-type', 'ivf'
    ],
    check=True,
    capture_output=False
)
print(f"\nReturn code: {result3.returncode}")

## Idempotency Audit Results

Should show PASS on all 5 audit checks: active_doc_count, active_chunk_count,
active_embedding_count, index_vector_count, chunk_hash_checksum.

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
df_audit = conn.execute("SELECT * FROM audit_results ORDER BY created_at DESC").df()
print(f"Audit results: {len(df_audit)} rows")
display(df_audit[['run_id_a', 'run_id_b', 'audit_type', 'result', 'details']])
conn.close()

if not df_audit.empty:
    pass_count = (df_audit['result'] == 'PASS').sum()
    fail_count = (df_audit['result'] == 'FAIL').sum()
    print(f"\nSummary: {pass_count} PASS, {fail_count} FAIL")

## STEP 4: Augmented Build

Run the augmented pipeline on `update_sample.jsonl` which contains:
- 1 **NEW** document (not previously seen)
- 1 **REVISED** document (same id, updated text → UPDATED)
- 1 **DUPLICATE** document (exact copy → UNCHANGED)
- 1 **CORRUPTED** document (empty text → REJECTED)

The FAISS index will be fully rebuilt from all active embeddings (new + existing).

In [ ]:
print("Running STEP 4: Augmented build on update_sample.jsonl...")
result4 = subprocess.run(
    [
        'python', 'src/build.py',
        '--mode', 'augmented',
        '--dataset-name', 'live_demo',
        '--input', 'data/live_demo/update_sample.jsonl',
        '--output', DEMO_OUTPUT,
        '--db', DEMO_DB,
        '--index-type', 'ivf'
    ],
    check=True,
    capture_output=False
)
print(f"\nReturn code: {result4.returncode}")

## Runs Table (All 3 Runs)

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
df_all_runs = conn.execute("SELECT * FROM runs ORDER BY run_id").df()
print(f"Total runs: {len(df_all_runs)}")
display(df_all_runs[[
    'run_id', 'mode', 'status',
    'raw_doc_count', 'new_doc_count', 'updated_doc_count',
    'unchanged_doc_count', 'rejected_doc_count',
    'active_doc_count', 'new_chunk_count', 'active_chunk_count',
    'new_embedding_count', 'active_embedding_count', 'index_vector_count'
]])
conn.close()

## Rejected Documents

Should show the corrupted doc with reason_code='EMPTY_TEXT'.

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
df_rejected = conn.execute(
    "SELECT doc_id, title, reason_code, reason_detail FROM rejected_documents"
).df()
print(f"Rejected documents: {len(df_rejected)}")
display(df_rejected)
conn.close()

## Row Count Reconciliation (Post Update)

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
df_recon2 = conn.execute(
    "SELECT * FROM row_count_reconciliation ORDER BY run_id"
).df()
print("Row count reconciliation (all runs):")
display(df_recon2)
conn.close()

## STEP 5: Retrieval After Update

Run the same queries again. The query about vector databases and retrieval systems
should now return the revised document (which had that text appended).

In [ ]:
from src.retrieve import retrieve

print("=== RETRIEVAL AFTER AUGMENTED UPDATE ===")
for q in pre_queries:
    print(f"\nQuery: {q}")
    df_r = retrieve(q, DEMO_DB, DEMO_INDEX, top_k=3)
    if df_r.empty:
        print("  No results.")
    else:
        for _, row in df_r.iterrows():
            print(f"  [{row['rank']}] {row['title']} (score={row['score']:.4f})")
            print(f"       {str(row['chunk_text'])[:120]}...")

print("\nNote: The 'vector databases and retrieval systems' query should now")
print(f"surface '{revised_title}' due to the text update.")

## Latency Logs (All Stages, All Runs)

In [ ]:
conn = duckdb.connect(DEMO_DB, read_only=True)
df_latency = conn.execute(
    """
    SELECT run_id, stage_name, duration_seconds, row_count
    FROM latency_logs
    ORDER BY run_id, start_time
    """
).df()
print(f"Latency records: {len(df_latency)}")
display(df_latency)

print("\nTotal time by run:")
display(df_latency.groupby('run_id')['duration_seconds'].sum().reset_index())
conn.close()

## Final Summary

### What We Demonstrated

This live demo showed the full lifecycle of the auditable Wikipedia RAG-ETL pipeline:

| Step | Action | Expected Behavior |
|------|--------|-------------------|
| 1 | Full build on 40 docs | All 40 → NEW, all chunks/embeddings created |
| 2 | Retrieval before update | Baseline results established |
| 3 | Full rebuild (idempotency) | All 40 → UNCHANGED, audit: all PASS |
| 4 | Augmented build | 1 NEW, 1 UPDATED, 1 UNCHANGED, 1 REJECTED |
| 5 | Retrieval after update | Revised doc surfaces for new query |

### Key Audit Properties Verified

1. **Idempotency**: Running the same full build twice produces identical state (audit PASS)
2. **SCD correctness**: Updated doc replaces old chunks/embeddings exactly
3. **Rejection tracking**: Corrupted doc logged with reason code
4. **FAISS rebuild**: Index always reflects current active embeddings only
5. **Row count reconciliation**: `active_chunks == active_embeddings == index_vectors`

### Monitoring Artifacts

- `runs` — pipeline run metadata
- `latency_logs` — per-stage timing
- `row_count_reconciliation` — cross-table count checks
- `audit_results` — cross-run idempotency checks
- `rejected_documents` — data quality tracking
- `faiss_index_registry` — index provenance